## TODO
* Автоэнкодер
* Cross-domain
* Beam search with lp and cp
* SRU
* Визуализация Attn
* replace_unk по attention'у http://opennmt.net/OpenNMT/translation/unknowns/

## Tutorials
* http://pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html
* https://github.com/spro/practical-pytorch/blob/master/seq2seq-translation/seq2seq-translation-batched.ipynb

## Articles
* Teaching neural networks to point to improve language modeling and translation: https://einstein.ai/research/teaching-neural-networks-to-point-to-improve-language-modeling-and-translation
* Training RNNs as Fast as CNNs : https://arxiv.org/abs/1709.02755
* Beam Search Strategies for Neural Machine Translation: https://arxiv.org/abs/1702.01806
* Unsupervised Machine Translation Using Monolingual Corpora Only: https://arxiv.org/abs/1711.00043
* Unsupervised Neural Machine Translation: https://arxiv.org/abs/1710.11041
* NIPS 2016 Tutorial: Generative Adversarial Networks: https://arxiv.org/pdf/1701.00160.pdf

## Repos
* https://github.com/facebookresearch/MUSE
* https://github.com/OpenNMT/OpenNMT-py

In [1]:
import torch
from src.train import Trainer

use_cuda = torch.cuda.is_available()
print(use_cuda)
%load_ext autoreload
%autoreload 2

True


In [2]:
SRC_LANG = "en"
TGT_LANG = "ru"
SRC_TO_TGT_DICT_FILENAME = "models/" + SRC_LANG + "-" + TGT_LANG + ".txt"
TGT_TO_SRC_DICT_FILENAME = "models/" + TGT_LANG + "-" + SRC_LANG + ".txt"
SRC_EMBEDDINGS = "models/wiki.multi." + SRC_LANG + ".vec"
TGT_EMBEDDINGS = "models/wiki.multi." + TGT_LANG + ".vec"
SRC_CORPUS = "data/corpus.tok.clean.tc." + SRC_LANG
TGT_CORPUS = "data/corpus.tok.clean.tc." + TGT_LANG

state = Trainer(SRC_LANG, TGT_LANG, use_cuda=use_cuda)
state.init_model([SRC_CORPUS, ], [TGT_CORPUS, ], SRC_EMBEDDINGS, TGT_EMBEDDINGS, SRC_TO_TGT_DICT_FILENAME,
                 TGT_TO_SRC_DICT_FILENAME, src_max_words=30000, tgt_max_words=40000, load_pretrained_embeddings=True, 
                 hidden_size=400)

corpus.tok.clean.tc.en:   2%|▏         | 1.05M/66.9M [00:00<00:07, 9.21MB/s]

corpus.tok.clean.tc.en: 100%|█████████▉| 66.9M/66.9M [00:06<00:00, 9.70MB/s]
corpus.tok.clean.tc.ru:  56%|█████▌    | 102M/181M [00:10<00:08, 9.15MB/s] 


Building model...
Loading embeddings...
UNMT(
  (encoder): EncoderRNN(
    (embedding): Embedding(70007, 300)
    (rnn): LSTM(300, 200, num_layers=3, dropout=0.1, bidirectional=True)
  )
  (decoder): AttnDecoderRNN(
    (embedding): Embedding(70007, 300)
    (attn): Attn(
      (attn): Linear(in_features=400, out_features=400)
      (sm): Softmax()
      (out): Linear(in_features=800, out_features=400)
      (tanh): Tanh()
    )
    (rnn): LSTM(700, 400, num_layers=3, dropout=0.1)
  )
  (src_generator): Generator(
    (out): Linear(in_features=400, out_features=30004)
    (sm): LogSoftmax()
  )
  (tgt_generator): Generator(
    (out): Linear(in_features=400, out_features=40004)
    (sm): LogSoftmax()
  )
  (discriminator): Discriminator(
    (layers): ModuleList(
      (0): Linear(in_features=20000, out_features=1024)
      (1): Linear(in_features=1024, out_features=1024)
      (2): Linear(in_features=1024, out_features=1024)
    )
    (out): Linear(in_features=1024, out_features=1)
  

In [ ]:
state.train([SRC_CORPUS, ], [TGT_CORPUS, ], big_epochs=3, batch_size=32, print_every=100)

In [3]:
state.load("model-0.pt")

In [ ]:
state.train_supervised([("data/parallel.tok.tc.en", "data/parallel.tok.tc.ru"), ],
                       big_epochs=5, batch_size=64, print_every=100)

parallel.tok.tc.en: 100%|█████████▉| 4.38M/4.39M [00:01<00:00, 2.83MB/s]
parallel.tok.tc.ru:  57%|█████▋    | 4.94M/8.73M [00:01<00:01, 3.04MB/s]


Batch: BilingualBatch: Variable containing:
     3      5    228  ...       9     91    451
   374   3414   4029  ...   11383    736     83
    13    138   3441  ...     387     13     13
        ...            ⋱           ...         
 10873      0      0  ...       0      0      0
     4      0      0  ...       0      0      0
     2      0      0  ...       0      0      0
[torch.LongTensor of size 44x64]
, Variable containing:
 30031  33063  30028  ...   30010  30688  30010
 30006  30028  30439  ...   30798  32438  30069
 30121  30161  30010  ...   32836  30354  33407
        ...            ⋱           ...         
 33035      0      0  ...       0      0      0
 30008      0      0  ...       0      0      0
 30005      0      0  ...       0      0      0
[torch.LongTensor of size 48x64]
, [44, 31, 31, 27, 27, 26, 26, 25, 25, 25, 24, 23, 23, 22, 21, 21, 21, 21, 19, 19, 19, 18, 18, 17, 17, 17, 16, 16, 16, 16, 15, 15, 15, 15, 14, 14, 14, 14, 13, 13, 13, 13, 13, 12, 12, 11, 11, 11, 

In [5]:
state.autoencode("my name is ilya .", "src")

'<unk> is <unk> km .'

In [6]:
state.autoencode("можно заказть фрукты .", "tgt")

'<unk> <unk> предоставляется <unk> .'

In [7]:
state.translate("all rooms have satellite TV and a private bathroom with soft bathrobes .", "src")

'во всех номерах есть телевизор и собственная ванная комната с халатами халатами .'

In [15]:
state.translate("можно заказать фрукты .", "tgt")

'could could they items etc . stuff . stuff . they could .'

In [9]:
with open("data/input.tok.tc.txt", "r", encoding='utf-8') as r, open("data/pred.txt", "w", encoding='utf-8') as w:
    for line in r:
        line = line.strip()
        translation = state.translate(line, "src")
        print(translation)
        w.write(translation+"\n")

в числе удобств телевизор с плоским экраном .
поездка до международного аэропорта <unk> занимает 20 минут .
в распоряжении гостей общий лаундж , общий лаундж и экскурсионное бюро .
из некоторых номеров открывается вид на горы .
за дополнительную плату предоставляется трансфер от / до аэропорта .
в отеле <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> есть открытый бассейн и гидромассажная ванна .
каждое утро для гостей сервируется завтрак .
отель <unk> <unk> <unk> <unk> <unk> расположен в центре города <unk> , в нескольких минутах ходьбы от <unk> <unk> и <unk> . к услугам гостей ресторан и рестораны .
в собственной ванной комнате с ванной установлена ванна и биде .
в собственной ванной комнате с душем предоставляются фен , бесплатные туалетно-косметические принадлежности и фен .
отель <unk> <unk> <unk> расположен всего в 100 метрах от пляжа <unk> , в <unk> км от центра города <unk> . к услугам гостей бесплатный <unk> , круглосуточная стойка регистрац

каждое утро в зале сервируют континентальный завтрак .
на территории обустроена бесплатная частная парковка .
расстояние до <unk> составляет <unk> км , а до <unk> <unk> — <unk> км .
на кухне установлена посудомоечная машина .
в номерах <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> есть кондиционер и телевизор с плоским экраном .
в <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
расстояние до аэропорта <unk> составляет 7 км .
к услугам гостей номера с бесплатным <unk> , бесплатный <unk> , бесплатная парковка , бизнес-центр и бесплатная парковка .
в ресторане <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> работают различные рестораны .
в некот

в некоторых номерах есть гостиный уголок .
в номерах есть кондиционер , телевизор , мини-бар , мини-бар и мини-бар .
гости могут воспользоваться в <unk> и <unk> на террасе .
в числе удобств номеров , в числе удобств — кондиционер , сейф , сейф и полностью оборудованная кухня .
отель находится в 5 минутах езды от пляжа <unk> и в 10 минутах езды от <unk> <unk> и <unk> <unk> .
в окрестностях можно взять напрокат велосипед , а также заняться <unk> .
в 5 минутах ходьбы от отеля <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> .
отель <unk> <unk> расположен в <unk> , в <unk> км от <unk> и в <unk> км от <unk> .
каждое утро в зале <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
поез

поездка до <unk> занимает 15 минут .
в апартаментах имеется собственная ванная комната , стиральная машина , фен и постельное белье .
к услугам гостей номера с кондиционером и бесплатным <unk> .
в отеле работает бесплатный <unk> . в отеле работает бесплатный <unk> .
в числе удобств — продуктовый магазин и ресторан .
апартаменты <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> расположен в городе <unk> , в 400 метрах от <unk> .
поездка от отеля до железнодорожного вокзала <unk> занимает 20 минут .
в ресторане <unk> <unk> <unk> <unk> <unk> подают блюда местной и интернациональной кухни .
на территории обустроена бесплатная частная парковка .
в апартаментах есть полностью оборудованная кухня , собственная ванная комната и телевизор с плоским экраном .
расстояние до <unk> составляет <unk> км , а до <unk> <unk> — <unk> км .
до центра города <unk> , <unk> <unk> , <unk> <unk> , <unk> <unk> , находится менее чем в 10 минутах ходьбы .
в числе удобств — круглосуточная стойка регистрации и прачечн

расстояние до международного аэропорта имени <unk> <unk> составляет 20 км .
в <unk> км находится <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> .
в ресторане отеля <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> .
расстояние до <unk> <unk> , <unk> <unk> составляет <unk> км , а до аэропорта <unk> можно доехать за 15 минут .
в 5 минутах ходьбы от отеля работают различные рестораны и рестораны .
отель <unk> <unk> <unk> <unk> <unk> расположен в <unk> , в <unk> км от <unk> <unk> и <unk> .
в номерах отеля <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> есть кондиционер , телевизор и мини-бар .
в некоторых номерах есть гостиная зона .
стойка регистрации отеля <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <u

...

за 5 минут можно дойти до центра <unk> и <unk> <unk> , откуда можно доехать до ближайшего <unk> .
отель <unk> <unk> <unk> находится в 5 минутах езды от <unk> <unk> и в 30 минутах езды от международного аэропорта <unk> .
в числе удобств — кондиционер , телевизор и собственная ванная комната .
в отеле <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> имеется собственная ванная комната и <unk> .
в <unk> км от апартаментов <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> — в 700 метрах .
расстояние от апартаментов <unk> до <unk> <unk> составляет <unk> км , а до <unk> — <unk> км .
прогулка до пляжа <unk> , <unk> <unk> <unk> занимает 5 минут .
гости могут заказать <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> 

из окон открывается вид на море .
дом для отпуска <unk> <unk> <unk> <unk> находится в <unk> <unk> .
из некоторых номеров открывается вид на море или сад .
в ресторане отеля <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> .
номера отеля <unk> <unk> <unk> <unk> оформлены в <unk> стиле .
в собственной ванной комнате установлен душ .
к услугам гостей открытый бассейн , гидромассажная ванна , принадлежности для барбекю и бесплатная парковка .
в некоторых номерах есть собственная , гидромассажная ванна или принадлежности .
в апартаментах есть кондиционер , кондиционер , балкон , полностью оборудованная кухня с телевизором и собственная ванная комната с душем и туалетом .
гости могут отдохнуть на террасе .
в числе удобств всех апартаментов — кондиционер , гостиный уголок и телевизор с плоским экраном .
в ресторане <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <

расстояние до ближайшего аэропорта <unk> составляет <unk> км .
поездка до <unk> занимает 20 минут .
отель <unk> <unk> <unk> расположен в городе <unk> , в 100 метрах от <unk> <unk> . к услугам гостей бесплатный <unk> и собственная ванная комната с собственной ванной комнатой .
в числе удобств — <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> — кухня с духовкой и микроволновой печью .
в ресторане <unk> <unk> <unk> <unk> <unk> <unk> сервируют завтрак « шведский стол » .
на территории обустроена бесплатная частная парковка .
в окрестностях можно заняться различными видами активного отдыха , как <unk> и <unk> .
до <unk> и железнодорожного вокзала <unk> можно доехать за 10 минут .
расстояние до <unk> <unk> <unk> <unk> составляет 5 км .
все номера оснащены кондиционером и телевизором .
поездка от отеля <unk> <unk> <unk> до <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> можн

из некоторых номеров открывается вид на горы .
поездка на автомобиле до города <unk> занимает 20 минут .
в отеле можно воспользоваться услугами <unk> .
отель <unk> <unk> расположен в тихом месте <unk> , в <unk> км от <unk> <unk> . к услугам гостей номера с кондиционером и бесплатным <unk> .
к услугам гостей сауна , гидромассажная ванна , сауна , а также номера с телевизором и бесплатным <unk> .
к услугам гостей бесплатный <unk> , прачечная и номера .
каждое утро в зале зале сервируется завтрак .
апартаменты <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> расположен в городе <unk> , в 5 минутах ходьбы от станции метро <unk> .
на территории обустроена бесплатная частная парковка .
номера оснащены телевизором с кабельными каналами и кондиционером .
каждое утро в обеденном зале отеля <unk> <unk> <unk> сервируется завтрак , а в баре можно заказать напитки и закуски в баре .
апартаменты <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk

к услугам гостей бесплатный <unk> и принадлежности для барбекю .
в гостевом доме работает пункт проката велосипедов .
все номера оснащены кондиционером , телевизором с плоским экраном и кабельными каналами .
в <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> — в городе .
в отеле <unk> <unk> сервируется завтрак « шведский стол » и в ресторане <unk> <unk> .
гостям предоставляются полотенца и постельное белье .
все номера отеля <unk> <unk> оформлены в современном стиле и оснащены кондиционером .
эти апартаменты с собственной кухней расположены в городе <unk> , в <unk> км от <unk> <unk> .
в ванной комнате установлен душ .
отель находится в 1,5 км от <unk> <unk> и <unk> , а также в 800 метрах от железнодорожного вокзала и железнодорожного вокзала .
на полностью оборудованной кухне установлены посудомоечная машина , микроволновая печь и плита .
в распоряжении гостей дома для отпуска <un

In [10]:
!perl multi-bleu.perl data/output.tok.tc.txt < data/pred.txt

BLEU = 12.36, 34.2/15.2/8.6/5.2 (BP=1.000, ratio=1.079, hyp_len=99629, ref_len=92344)
It is in-advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.
